In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
import numpy as np
import pandas as pd
from sklearn.preprocessing import LabelEncoder
import joblib
from torch.utils.data import TensorDataset, DataLoader

# Laden des Datensatzes
df = pd.read_csv('Datasets/Iris.csv')

# 2. Features (X) und Target (y) trennen
X = df.drop('species', axis=1).values
y = df['species'].values
# LabelEncoder sortiert die gefundenen Text-Kategorien standardmäßig alphabetisch
le = LabelEncoder()
y = le.fit_transform(y)
 
# Aufteilen des Datensatzes in Trainings- und Testdaten
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
X_train, X_val, y_train, y_val = train_test_split(X_train, y_train, test_size=0.2, random_state=0)
 
# Umwandeln der Daten in PyTorch-Tensoren
# Hinweis: from_numpy() teilt Speicher mit NumPy (keine Kopie) – effizienter als torch.tensor()
X_train = torch.from_numpy(X_train).float()
X_test  = torch.from_numpy(X_test).float()
y_train = torch.from_numpy(y_train).long()
y_test  = torch.from_numpy(y_test).long()
X_val   = torch.from_numpy(X_val).float()
y_val   = torch.from_numpy(y_val).long()

# -- NEU: 
# Erstellen eines TensorDatasets und DataLoaders - Kombinieren von Features und Labels zu einem Dataset
train_dataset = TensorDataset(X_train, y_train)
# Erstellen des DataLoaders für das Training
batch_size = 12
# shuffle=True sorgt dafür, dass die Daten in jeder Epoche gemischt werden
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
# -- Ende NEU --

# Definieren des neuronalen Netzes als Sequential-Modell
net = nn.Sequential(
    nn.Linear(4, 5),   # Eingabeschicht (4 Merkmale) -> Versteckte Schicht (10 Neuronen)
    nn.ReLU(),           # Aktivierungsfunktion: ReLU
    nn.Linear(5, 3)     # Versteckte Schicht (10 Neuronen) -> Ausgabeschicht (3 Klassen)
)
 
# Definieren des Verlustkriteriums und des Optimierers
criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(net.parameters(), lr=0.01)
 
# Trainieren des neuronalen Netzes
net.train()  # Trainingsmodus aktivieren
for epoch in range(100):
    for batch_data in train_loader: # Schleife über die Batches aus dem DataLoader
        batch_X = batch_data[0]
        batch_y = batch_data[1]
        optimizer.zero_grad()
        outputs = net(batch_X)
        loss = criterion(outputs, batch_y)
        loss.backward()
        optimizer.step()
 
    # Validierung nach jeder Epoche
    net.eval()
    with torch.no_grad():
        val_out  = net(X_val)
        val_loss = criterion(val_out, y_val)
    net.train()
 
    print(f'Epoch {epoch:3d} | Loss: {loss.item():.4f}')
 
# Auswerten des neuronalen Netzes auf den Testdaten
net.eval()
with torch.no_grad():
    outputs = net(X_test)
    _, predicted = torch.max(outputs, 1)
    accuracy = accuracy_score(y_test, predicted)
    print('Testgenauigkeit: ', accuracy)
 
# Abspeichern des trainierten Netzes
torch.save(net.state_dict(), 'Models/iris_net_NextStep1.pth')
joblib.dump(le, 'Models/label_encoder_NextStep1.pkl')

In [ ]:
# Wiederladen des State-Dicts
import torch
import torch.nn as nn

net.load_state_dict(torch.load('Models/iris_net_NextStep1.pth', map_location=torch.device('cpu')))
le = joblib.load('Models/label_encoder_NextStep1.pkl')

net = nn.Sequential(
    nn.Linear(4, 5),    # Eingabeschicht (4 Merkmale) -> Versteckte Schicht (10 Neuronen)
    nn.ReLU(),          # Aktivierungsfunktion: ReLU
    nn.Linear(5, 3)     # Versteckte Schicht (10 Neuronen) -> Ausgabeschicht (3 Klassen)
)
for k, v in net.named_parameters():
    print(k,v)
net.eval()

In [ ]:
# Vorhersage mit dem trainierten Netz

# Eingabe der vier Merkmale
print("Geben Sie die vier Merkmale ein:")
sepal_length = float(input("Sepal-Länge (cm): "))
sepal_width = float(input("Sepal-Breite (cm): "))
petal_length = float(input("Petal-Länge (cm): "))
petal_width = float(input("Petal-Breite (cm): "))

# Erstellen eines Tensors aus den Eingabewerten
inputs = torch.tensor([[sepal_length, sepal_width, petal_length, petal_width]], dtype=torch.float32)

# Vorhersage treffen
with torch.no_grad():
    outputs = net(inputs)
    _, predicted = torch.max(outputs, 1)

# Ausgabe der Klassifizierungsaussage
print("Klasse: ", le.inverse_transform([predicted.item()])[0])

In [ ]:
# aus dem obigen Skript wird inputs übernommen, 
# die Vorhersagen werden als Wahrscheinlichkeiten ausgegeben

import torch.nn.functional as F

# Vorhersage des Netzes
with torch.no_grad():
    output = net(inputs)
    _, max_index = torch.max(output, 1)
    print("Vorhergesagte Klasse:", max_index.item())

# Klassenwahrscheinlichkeiten berechnen
probabilities = F.softmax(output, dim=1)
print("Klassenwahrscheinlichkeiten:", probabilities.numpy()[0])